In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"E:\404_Found\H&E Stained Histological Nucleus Instancve Segmentation by YOLOv8x\PanNuke_Results\Dump_Seg_Results\nuclei_analysis.csv")

In [ ]:
# we r putting thresholds for density; 

low_density = df["density"].quantile(0.25) # quantile(0.25) ---> if density is below 25% then low_density, better than hardcoded values eg, if density < 0.03, then  low_density; we use this type of percentile based quantifers as this adapts automatically and works on every dataset; 

high_density = df["density"].quantile(0.75)

In [4]:
# we r putting thresholds for avg_norm_area; 

small_size = df["avg_norm_area"].quantile(0.25) 
large_size = df["avg_norm_area"].quantile(0.75)

# the range : 
    # ≤ 25%	    small
    # ≥ 75%	    large

In [ ]:
# density classification function; it takes the value(density) and decides(labels) it is low or high or medium;
# continuous value → categorical label

def classify_density(x):
    if x <= low_density:
        return "Low Density" # bottom 25% ---> Low;
    elif x >= high_density:
        return "High Density" # top 25% ---> High;
    else:
        return "Medium Density" # Everything in between ---> Medium;

df["density_label"] = df["density"].apply(classify_density) # apply func : it runs the function on every row(internally using a for loop for each and every row);
# new column(density_label) created;

In [ ]:
# same func for sizes;

def classify_size(x):
    if x <= small_size:
        return "Small Nuclei"
    elif x >= large_size:
        return "Large Nuclei"
    else:
        return "Medium Nuclei"

df["size_label"] = df["avg_norm_area"].apply(classify_size) # new column(size_label) created;

In [7]:
# now combining both newly created columns(density_label and size_label) and creating a new column(tissue_profile);
# this is feature combination ---> semantic interpretation

def profile(row): # input ---> entire row
    if row["density_label"] == "High Density" and row["size_label"] == "Small Nuclei":
        return "Crowded & Small Cells"
    
    elif row["density_label"] == "High Density" and row["size_label"] == "Large Nuclei":
        return "Dense Tissue with Large Cells"
    
    elif row["density_label"] == "Low Density" and row["size_label"] == "Large Nuclei":
        return "Sparse Tissue with Enlarged Cells"
    
    elif row["density_label"] == "Low Density" and row["size_label"] == "Small Nuclei":
        return "Sparse & Uniform Tissue"
    
    else:
        return "Moderate Tissue Structure"

df["tissue_profile"] = df.apply(profile, axis=1)

In [8]:
df[["image_name", "density_label", "size_label", "tissue_profile"]].head()

,image_name,density_label,size_label,tissue_profile
0,1-100.png,Medium Density,Medium Nuclei,Moderate Tissue Structure
1,1-1000.png,Low Density,Small Nuclei,Sparse & Uniform Tissue
2,1-1009.png,Medium Density,Medium Nuclei,Moderate Tissue Structure
3,1-1012.png,Low Density,Medium Nuclei,Moderate Tissue Structure
4,1-1020.png,High Density,Small Nuclei,Crowded & Small Cells


In [ ]:
# distribution of Tissue Types : 
 
df["tissue_profile"].value_counts()

tissue_profile
Moderate Tissue Structure            855
Sparse & Uniform Tissue              124
Crowded & Small Cells                 93
Sparse Tissue with Enlarged Cells     82
Dense Tissue with Large Cells         32
Name: count, dtype: int64

In [10]:
# Now Saving this info in a seperate CSV file for more structured DS;

profile_df = df[[
    "image_name",
    "density_label",
    "size_label",
    "tissue_profile"
]].copy() # .copy() ---> safe memory, NO linkage;

profile_df.head()

,image_name,density_label,size_label,tissue_profile
0,1-100.png,Medium Density,Medium Nuclei,Moderate Tissue Structure
1,1-1000.png,Low Density,Small Nuclei,Sparse & Uniform Tissue
2,1-1009.png,Medium Density,Medium Nuclei,Moderate Tissue Structure
3,1-1012.png,Low Density,Medium Nuclei,Moderate Tissue Structure
4,1-1020.png,High Density,Small Nuclei,Crowded & Small Cells


In [11]:
profile_df["tissue_profile"].value_counts() # just a sanity check that everything working good on the newly craeted CSV file;

tissue_profile
Moderate Tissue Structure            855
Sparse & Uniform Tissue              124
Crowded & Small Cells                 93
Sparse Tissue with Enlarged Cells     82
Dense Tissue with Large Cells         32
Name: count, dtype: int64

In [12]:
# saving that new CSV file; 
profile_df.to_csv(r"E:\404_Found\H&E Stained Histological Nucleus Instancve Segmentation by YOLOv8x\PanNuke_Results\Dump_Seg_Results\tissue_profiles.csv", index=False)